# Interactive Maps with Folium
Visualizing SpaceX launch sites on an interactive map

In [1]:
import pandas as pd
import folium
from folium import plugins

In [2]:
df = pd.read_csv('spacex_launches_raw.csv')

# Define site coordinates
sites = {
    'CCAFS LC-40': (28.5621, -80.5774),
    'VAFB SLC-4E': (34.6321, -120.6270),
    'KSC LC-39A': (28.6084, -80.6037),
    'Boca Chica': (25.9972, -97.1589)
}

# Calculate launch counts by site
site_counts = df['Launch_Site'].value_counts()
site_success = df.groupby('Launch_Site')['Class'].agg(['sum', 'count'])
site_success['rate'] = site_success['sum'] / site_success['count']

### Create Interactive Map

In [3]:
# Create base map centered on US
m = folium.Map(location=[39.8283, -98.5795], zoom_start=4)

# Add markers for each site
for site_name, (lat, lon) in sites.items():
    success_count = site_success.loc[site_name, 'sum']
    total_count = site_success.loc[site_name, 'count']
    success_rate = site_success.loc[site_name, 'rate']
    
    # Color based on success rate
    color = 'green' if success_rate >= 0.75 else 'orange' if success_rate >= 0.5 else 'red'
    
    popup_text = f"""<b>{site_name}</b><br>
    Launches: {total_count}<br>
    Successful: {int(success_count)}<br>
    Success Rate: {success_rate*100:.1f}%"""
    
    folium.CircleMarker(
        location=[lat, lon],
        radius=total_count/3,
        popup=popup_text,
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.7
    ).add_to(m)

# Save map
m.save('spacex_map.html')
print(f"Map created with 4 launch sites")

### Map Features:
- **Green circles**: Launch sites with 75%+ success rate
- **Orange circles**: Launch sites with 50-75% success rate
- **Red circles**: Launch sites with <50% success rate
- **Circle size**: Proportional to number of launches
- **Click for details**: Success statistics for each site

In [4]:
print("Site Summary:")
for site in sorted(sites.keys()):
    count = site_success.loc[site, 'count']
    rate = site_success.loc[site, 'rate']
    print(f"{site}: {int(count)} launches ({rate*100:.0f}% success)")